# Cluster each run & extract per-cluster CDR3 motifs

For every TCR design **run** (an `(experiment, run)` group in `figures/data.csv`) this notebook
clusters the in-pool designs by CDR3 sequence (normalized Levenshtein distance + average-linkage
hierarchical clustering, via `utils.cluster_seqs`) and renders a sequence **motif** (mafft MSA +
logomaker, via `utils.get_logo`) for each cluster.

Clustering is done on three CDR3 sets: alpha (`a`), beta (`b`) and joint alpha+beta (`ab`).
Outputs: per-cluster motif figures in `figures/cluster_motifs/` and `figures/run_clustered_data.csv`
with the `cluster_a/b/ab` assignments.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from utils import cluster_seqs, get_logo

plt.rcParams['font.family'] = 'serif'

# CDR3 set -> column(s) to cluster/visualise on
CDR_SETS = {
    'a':  ['acdr3'],            # alpha CDR3
    'b':  ['bcdr3'],            # beta CDR3
    'ab': ['acdr3', 'bcdr3'],   # joint alpha+beta CDR3
}
THRESHOLD = 0.7   # edit-distance cut height for fcluster (lower => more, tighter clusters)

def _joined(g, cols):
    """Concatenate CDR3 column(s) row-wise into one string per design."""
    return g[cols].astype(str).agg(''.join, axis=1).values

In [ ]:
df = pd.read_csv('figures/data.csv')
pool = df[df['in_pool'].astype(bool)].copy()
print(f'{len(pool)} in-pool designs across {pool.groupby(["experiment","run"]).ngroups} runs')
pool.groupby(['experiment', 'run']).size()

## Cluster within each run
Adds `cluster_a/b/ab` columns, clustering each run independently.

In [ ]:
def cluster_runs(df, threshold=THRESHOLD, pool_only=True):
    df = df.copy()
    for key in CDR_SETS:
        df[f'cluster_{key}'] = pd.NA
    sub = df[df['in_pool'].astype(bool)] if pool_only else df
    for (exp, run), g in sub.groupby(['experiment', 'run']):
        for key, cols in CDR_SETS.items():
            seqs = list(_joined(g, cols))
            labels = cluster_seqs(seqs, threshold=threshold) if len(seqs) > 1 else np.ones(len(seqs), int)
            df.loc[g.index, f'cluster_{key}'] = labels
    return df

df = cluster_runs(df)
pool = df[df['in_pool'].astype(bool)]
summary = pool.groupby(['experiment', 'run']).agg(
    n=('design', 'size'),
    n_clust_a=('cluster_a', 'nunique'),
    n_clust_b=('cluster_b', 'nunique'),
    n_clust_ab=('cluster_ab', 'nunique'),
)
df.to_csv('figures/run_clustered_data.csv')
summary

## Motif per cluster
One stacked sequence logo per cluster, for a chosen run and CDR3 set.

In [ ]:
def motif_figure(g, key, score_col='score'):
    cols, ccol = CDR_SETS[key], f'cluster_{key}'
    clusters = sorted(pd.unique(g[ccol].dropna()))
    fig, axes = plt.subplots(nrows=len(clusters), figsize=(8, max(2, 2.2 * len(clusters))), squeeze=False)
    for c, ax in zip(clusters, axes[:, 0]):
        rows = g[g[ccol] == c]
        seqs = [str(s) for s in _joined(rows, cols)]
        try:
            get_logo(seqs, color_scheme='hydrophobicity', ax=ax)
        except Exception as ex:
            ax.text(0.5, 0.5, f'logo failed: {ex}', ha='center')
        ax.set_ylabel('count')
        ax.set_title(f'cluster {int(c)} - n={len(rows)}, mean {score_col}={rows[score_col].mean():.2f}', fontsize=9)
    fig.tight_layout()
    return fig

# Preview: joint alpha+beta motifs for one run
exp, run, key = 'full_run', 4, 'ab'
g = pool[(pool['experiment'] == exp) & (pool['run'] == run)]
fig = motif_figure(g, key)
fig.suptitle(f'{exp} / run {run} - CDR3{key} motifs', y=1.002)
plt.show()

## Export all runs
Writes a motif figure for every run x CDR3 set into `figures/cluster_motifs/`.

In [ ]:
out_dir = Path('figures/cluster_motifs'); out_dir.mkdir(exist_ok=True)
for (exp, run), g in pool.groupby(['experiment', 'run']):
    tag = f'{exp}_run{run}'.replace('*', 'x').replace('/', '_')
    for key in CDR_SETS:
        fig = motif_figure(g, key)
        fig.suptitle(f'{exp} / run {run} - CDR3{key} motifs', y=1.002)
        fig.savefig(out_dir / f'{tag}_cdr3{key}.pdf', bbox_inches='tight')
        plt.close(fig)
print('Saved', len(list(out_dir.glob('*.pdf'))), 'figures to', out_dir)